# NAS - Top-K Beam Selection

- **Authored by:** Matheus Ferreira Silva 
- **GitHub:**: https://github.com/MatheusFS-dev

### **Data Summary**

- **beam_output**  
  - **Shape:** (9638, 8, 32)  
  - **Represents:** Target labels representing beam scores for each sample.  
  - **Example:** For one sample, a portion of the beam scores might look like:  
    ```
    [[1.51e-06, 1.76e-06, 3.01e-06, ...], 
     [1.77e-06, 2.02e-06, 3.37e-06, ...], 
     ...]
    ```  
    Each value is a float (with an imaginary part of zero), indicating the quality of a specific beam pair.

- **coord_input**  
  - **Shape:** (9638, 2)  
  - **Represents:** 2D coordinates associated with each sample (e.g., spatial positions).  
  - **Example:** The first sample might have coordinates similar to:  
    ```
    [748.92, 624.72]
    ```

- **image_input**  
  - **Shape:** (9638, 48, 81, 1)  
  - **Represents:** Grayscale image data where each pixel is an 8-bit unsigned integer.  
  - **Example:** A snippet from the first image might include pixel values such as:  
    ```
    [[[156], [136], [119], ...],
     [[131], [105], [91], ...],
     [[153], [116], [100], ...],
     ...]
    ```

- **lidar_input**  
  - **Shape:** (9638, 20, 200, 10)  
  - **Represents:** LIDAR data formatted as a multi-channel grid (20×200 with 10 channels), encoding spatial features or intensities. This representation indicates that LIDAR data contains spatial information about obstructions, base stations, and the target vehicle, which can be crucial for predicting beamforming paths.
  - **Semantic Meaning of Voxel Values:**  
    - **-2:** Base Station (BS) location  
    - **-1:** Target vehicle (receiver)  
    - **1:** Obstacles (e.g., other vehicles, buildings, pedestrians, trees)  
    - **0:** Empty space (free path for mmWave signals)  
  - **Example:**  
    ```
    [[[0, 0, 0, ..., 0, 0, 0],
      [0, 0, 0, ..., 0, 0, 0],
      ...,
      [-2, -1, 1, ..., 0, 0, 0]],
     ...]
    ```

## 1. Imports

In [ ]:
%pip install numpy scipy scikit-learn optuna

# --------------------------- Standard Libraries ---------------------------- #
import os
import gc
import traceback
from IPython.display import clear_output, display, HTML

# -------------------------------- Annotations ------------------------------- #
from typing import *

# ------------------------- Data Processing Libraries ----------------------- #
import numpy as np
import scipy.io
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# ----------------------- TensorFlow and Keras Modules ---------------------- #
import tensorflow as tf
from tensorflow.keras.backend import clear_session
from tensorflow.keras import (
    layers, 
    Model, 
    callbacks, 
    optimizers, 
    regularizers, 
    mixed_precision,
    losses,
)
from tensorflow.python.profiler.model_analyzer import profile
from tensorflow.python.profiler.option_builder import ProfileOptionBuilder

tf.get_logger().setLevel('ERROR')

# ---------------------------- Scikit-learn Modules ------------------------- #
from sklearn.preprocessing import (
    MinMaxScaler,
    StandardScaler,
    RobustScaler,
    QuantileTransformer,
    PowerTransformer,
)
from sklearn.model_selection import train_test_split
from sklearn.decomposition import PCA

# ---------------------------------- Optuna ---------------------------------- #
import optuna
# import optunahub # For the AutoSampler

# ------------------------------- Local Import ------------------------------- #
from utils.email_api import run_with_notification

## 2. Utility Functions Definitions

### 2.1. GPU

In [ ]:
def get_gpu_info():
    """
    Retrieves and prints detailed GPU information including TensorFlow,
    CUDA, cuDNN versions, number of GPUs, and memory details.
    """
    # Display TensorFlow version
    print(f"TensorFlow Version: {tf.__version__}")

    # Check if TensorFlow is built with CUDA support and retrieve build info
    if tf.test.is_built_with_cuda():
        build_info = tf.sysconfig.get_build_info()
        print(f"TensorFlow is built with CUDA support")
        print(f"CUDA Version: {build_info['cuda_version']}")
        print(f"cuDNN Version: {build_info['cudnn_version']}")
    else:
        print("Running on CPU (No CUDA support detected)")

    # Detect available GPUs
    gpus = tf.config.list_physical_devices("GPU")
    if gpus:
        print(f"\nNumber of GPUs detected: {len(gpus)}")
        print(f"Available GPU(s): {[gpu.name for gpu in gpus]}\n")
        tf.test.gpu_device_name()
    else:
        print("No GPUs found")
        print("Running on CPU")

### 2.2. Folder and Files

In [ ]:
def create_run_directory(prefix: str, base_dir: str = "runs") -> str:
    """
    Creates a new directory for storing training logs, checkpoints, and plots.
    The directory name is based on the next available number.

    Args:
        prefix (str): Prefix for the run directory.
        base_dir (str): Base directory for storing training runs. Defaults to "runs".

    Returns:
        str: Path to the created run directory.
    """
    os.makedirs(base_dir, exist_ok=True)  # Ensure the base directory exists

    # Find the next available run number
    existing_dirs = [d for d in os.listdir(base_dir) if d.startswith(prefix) and d[len(prefix):].isdigit()]
    next_run_number = max([int(d[len(prefix):]) for d in existing_dirs] + [0]) + 1
    run_dir = os.path.join(base_dir, f"{prefix}{next_run_number}")
    os.makedirs(run_dir, exist_ok=True)  # Create the run directory

    return run_dir

In [ ]:
def save_trial_params_to_file(filepath: str, params: dict, **kwargs) -> None:
    """
    Saves the parameters of a trial along with additional information to a file.

    Args:
        filepath (str): Path to the file where parameters will be saved.
        params (dict): Parameters of the trial to save.
        **kwargs: Additional information to include in the file (e.g., rank, trial ID, loss).
    """
    with open(filepath, "w") as txt_file:
        # Write additional info
        for key, value in kwargs.items():
            txt_file.write(f"{key}: {value}\n")
        
        # Write parameters
        txt_file.write("Parameters:\n")
        for param_name, param_value in params.items():
            txt_file.write(f"  {param_name}: {param_value}\n")

In [ ]:
def analyze_study(study: optuna.Study, fig_dir: str) -> None:
    """
    Analyzes an Optuna study: counts failed trials, summarizes hyperparameters,
    and plots correlations.

    Args:
        study (optuna.Study): The completed Optuna study object.
        fig_dir (str): Directory to save plots and summary tables.
    """
    clear_output(wait=True)
    os.makedirs(fig_dir, exist_ok=True)

    # ————————————— Count Failed Trials ————————————— #
    failed = sum(1 for t in study.trials if t.state != optuna.trial.TrialState.COMPLETE)
    print(f"Number of failed trials: {failed}\n")

    # ——————— Load & Filter Completed Trials ——————— #
    df = study.trials_dataframe().query("state == 'COMPLETE'").copy()
    if df.empty:
        print("No completed trials to analyze.")
        return
    df.drop(columns=["number", "datetime_start", "datetime_complete", "state", "duration"],
            errors="ignore", inplace=True)

    # ————— Identify Numeric vs Categorical ————— #
    num_cols = df.select_dtypes(include="number").columns.tolist()
    cat_cols = df.select_dtypes(include="object").columns.tolist()

    # ————— Numeric Summary (Mean, Std, Variance, Range) ————— #
    numeric_summary = []
    for col in num_cols:
        arr = df[col].dropna()
        numeric_summary.append({
            "Parameter": col,
            "Mean": arr.mean(),
            "Std": arr.std(),
            "Variance": arr.var(),
            "Range": f"{arr.min():.5g} to {arr.max():.5g}"
        })
    numeric_df = pd.DataFrame(numeric_summary)

    display(HTML("<h3>Numeric Hyperparameter Summary</h3>"))
    display(numeric_df)
    
    # ————— Categorical Counts Table ————— #
    cat_rows = []
    for col in cat_cols:
        counts = df[col].value_counts()
        for cat, cnt in counts.items():
            cat_rows.append({
                "Parameter": col,
                "Category": cat,
                "Count": int(cnt)
            })
    categorical_df = pd.DataFrame(cat_rows)
    display(HTML("<h3>Categorical Hyperparameter Counts</h3>"))
    display(categorical_df)

    # ————— Numeric Histograms ————— #
    if num_cols:
        n = len(num_cols)
        ncols = min(n, 3)
        nrows = math.ceil(n / ncols)
        fig, axes = plt.subplots(nrows, ncols, figsize=(8 * ncols, 6 * nrows))
        axes = axes.flatten()
        for idx, col in enumerate(num_cols):
            ax = axes[idx]
            sns.histplot(df[col].dropna(), kde=False, ax=ax)
            ax.set_title(col)
            ax.set_xlabel(col)
            ax.set_ylabel("Frequency")
        # remove extra axes
        for idx in range(len(num_cols), len(axes)):
            fig.delaxes(axes[idx])
        plt.tight_layout()
        fig.savefig(os.path.join(fig_dir, "numeric_histograms.png"))
        plt.show()
        plt.close(fig)

    # ————— Scatter vs Objective ————— #
    if "value" in df.columns and num_cols:
        scatter_cols = [c for c in num_cols if c != "value"]
        if scatter_cols:
            n = len(scatter_cols)
            ncols = min(n, 3)
            nrows = math.ceil(n / ncols)
            fig, axes = plt.subplots(nrows, ncols, figsize=(8 * ncols, 6 * nrows))
            axes = axes.flatten()
            for idx, col in enumerate(scatter_cols):
                ax = axes[idx]
                ax.scatter(df[col], df["value"], alpha=0.6, s=10)
                ax.set_title(f"{col} vs Objective")
                ax.set_xlabel(col)
                ax.set_ylabel("Objective Value")
                ax.grid(True)
            for idx in range(len(scatter_cols), len(axes)):
                fig.delaxes(axes[idx])
            plt.tight_layout()
            fig.savefig(os.path.join(fig_dir, "numeric_scatter_vs_objective.png"))
            plt.show()
            plt.close(fig)

    # ————— Categorical Frequency Bars ————— #
    if cat_cols:
        n = len(cat_cols)
        ncols = min(n, 3)
        nrows = math.ceil(n / ncols)
        fig, axes = plt.subplots(nrows, ncols, figsize=(8 * ncols, 6 * nrows))
        axes = axes.flatten()
        for idx, col in enumerate(cat_cols):
            ax = axes[idx]
            counts = df[col].value_counts()
            sns.barplot(
                x=counts.index.astype(str),
                y=counts.values,
                ax=ax
            )
            ax.set_title(col)
            ax.set_ylabel("Count")
            ax.set_xlabel("")
            # Rotate tick labels for readability
            plt.sca(ax)
            plt.xticks(rotation=45, ha="right")
        for idx in range(len(cat_cols), len(axes)):
            fig.delaxes(axes[idx])
        plt.tight_layout()
        fig.savefig(os.path.join(fig_dir, "categorical_frequencies.png"))
        plt.show()
        plt.close(fig)


## 3. Setup and Configuration

### 3.1. GPU Management

In [ ]:
# Specify GPU to use (e.g., GPU 0)
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
get_gpu_info()

### 3.2. Random Seed

In [ ]:
SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

### 3.3. Run Directory 

In [ ]:
# Set to an existing path to resume training
RESUME_TRAINING_PATH = "runs/nas_1"  # "runs/nas_1" 

RUN_DIR = RESUME_TRAINING_PATH or create_run_directory(prefix="nas_")

print(f"Run directory: {RUN_DIR}")

## 4. Data Loading and Preprocessing

### 4.1. Data Loading

In [ ]:
def convert_to_sparse_labels(y: np.ndarray) -> np.ndarray:
    """
    Converts beam score targets to sparse integer labels suitable for SparseCategoricalCrossentropy loss.

    Args:
        y (np.ndarray): Original beam scores with shape (N, 8, 32).

    Returns:
        np.ndarray: Array of integer labels with shape (N,), where each label corresponds 
                    to the index (flattened over 8x32) of the maximum score.
    """
    # Reshape the input so that each sample becomes a 1D array (e.g., 256 elements)
    y_flat = y.reshape(y.shape[0], -1)
    # For each sample, return the index of the maximum value
    labels = np.argmax(y_flat, axis=1)
    return labels


In [ ]:
# Define the base directory for data files
DATA_DIR = "./data/s008"

# Construct full paths to the .npy files
beam_output_path = os.path.join(DATA_DIR, "beam_output", "output_classification.npy")
coord_input_path = os.path.join(DATA_DIR, "coord_input", "coordinates.npy")
image_input_path = os.path.join(DATA_DIR, "image_input", "inputs.npy")
lidar_input_path = os.path.join(DATA_DIR, "lidar_input", "input.npy")

# Load the data from the .npy files
s008_y_train = np.load(beam_output_path)
s008_coord_input = np.load(coord_input_path)
s008_image_input = np.load(image_input_path)
s008_lidar_input = np.load(lidar_input_path)

# Cast target beam outputs to float - REMOVING USELESS IMAG PART
s008_y_train = s008_y_train.astype(np.float32)
s008_coord_input = s008_coord_input.astype(np.float32)

print(f"Shape before conversion: {s008_y_train.shape}")
s008_y_train = convert_to_sparse_labels(s008_y_train)
print(f"Shape after conversion: {s008_y_train.shape}")

# Print the shapes of the loaded data
print(f"y_train shape: {s008_y_train.shape}")
print(f"coord_input shape: {s008_coord_input.shape}")
print(f"image_input shape: {s008_image_input.shape}")
print(f"lidar_input shape: {s008_lidar_input.shape}")

In [ ]:
# Define the base directory for data files
DATA_DIR = "./data/s009"

# Construct full paths to the .npy files
beam_output_path = os.path.join(DATA_DIR, "beam_output", "output_classification.npy")
coord_input_path = os.path.join(DATA_DIR, "coord_input", "coordinates.npy")
image_input_path = os.path.join(DATA_DIR, "image_input", "inputs.npy")
lidar_input_path = os.path.join(DATA_DIR, "lidar_input", "input.npy")

# Load the data from the .npy files
s009_y_train = np.load(beam_output_path)
s009_coord_input = np.load(coord_input_path)
s009_image_input = np.load(image_input_path)
s009_lidar_input = np.load(lidar_input_path)

# Cast target beam outputs to float - REMOVING USELESS IMAG PART
s009_y_train = s009_y_train.astype(np.float32)
s009_coord_input = s009_coord_input.astype(np.float32)

print(f"Shape before conversion: {s009_y_train.shape}")
s009_y_train = convert_to_sparse_labels(s009_y_train)
print(f"Shape after conversion: {s009_y_train.shape}")

# Print the shapes of the loaded data
print(f"y_train shape: {s009_y_train.shape}")
print(f"coord_input shape: {s009_coord_input.shape}")
print(f"image_input shape: {s009_image_input.shape}")
print(f"lidar_input shape: {s009_lidar_input.shape}")

In [ ]:
# # ----------------------- Subsample dataset for testing ---------------------- #
# SAMPLE_SIZE = 50  # Use a subset of x samples for testing
# s008_y_train = s008_y_train[:SAMPLE_SIZE]
# s008_coord_input = s008_coord_input[:SAMPLE_SIZE]
# s008_image_input = s008_image_input[:SAMPLE_SIZE]
# s008_lidar_input = s008_lidar_input[:SAMPLE_SIZE]

# s009_y_train = s009_y_train[:SAMPLE_SIZE]
# s009_coord_input = s009_coord_input[:SAMPLE_SIZE]
# s009_image_input = s009_image_input[:SAMPLE_SIZE]
# s009_lidar_input = s009_lidar_input[:SAMPLE_SIZE]

# # Print the shapes of the loaded data
# print(f"y_train s008 shape: {s008_y_train.shape}")
# print(f"coord_input s008 shape: {s008_coord_input.shape}")
# print(f"image_input s008 shape: {s008_image_input.shape}")
# print(f"lidar_input s008 shape: {s008_lidar_input.shape}")

# print(f"y_train s009 shape: {s009_y_train.shape}")
# print(f"coord_input s009 shape: {s009_coord_input.shape}")
# print(f"image_input s009 shape: {s009_image_input.shape}")
# print(f"lidar_input s009 shape: {s009_lidar_input.shape}")

## 5. Objective Function Definition

In [ ]:
def get_regularizer(trial: optuna.Trial, name: str):
    """Returns the selected regularizer based on the trial suggestion."""
    reg_type = trial.suggest_categorical(
        name,
        [
            "none",
            "l1",
            "l2",
            "l1l2",
            # "orthogonal", #! For rank-2 tensors only
        ],
    )

    if reg_type == "l1":
        return regularizers.L1(l1=0.01)
    elif reg_type == "l2":
        return regularizers.L2(l2=0.01)
    elif reg_type == "l1l2":
        return regularizers.L1L2(l1=0.01, l2=0.01)
    elif reg_type == "orthogonal":
        # Apply OrthogonalRegularizer only if units > 1
        return regularizers.OrthogonalRegularizer(
            factor=0.01,
            mode="rows", # "rows" or "columns"
        )
    else:
        return None  # No regularization

Below is a table summarizing the four regularizers:

| Regularizer             | What It Does                                                                                              | When to Use                                                                                   | Benefits                                                                                                  | Disadvantages                                                              |
|-------------------------|-----------------------------------------------------------------------------------------------------------|----------------------------------------------------------------------------------------------|-----------------------------------------------------------------------------------------------------------|-----------------------------------------------------------------------------|
| **L1**                  | Adds the sum of the absolute values of weights (‖w‖₁) to the loss, encouraging many weights to become zero.  | Use when sparsity is desired, especially in high-dimensional settings with many irrelevant features.  | Promotes sparsity, acts as feature selection, and can lead to more interpretable models.                 | Non-smooth gradients at zero; may cause optimization instability.          |
| **L2**                  | Adds the sum of squared weights (‖w‖₂²) to the loss, discouraging large weights via quadratic penalization.  | Common default in neural networks to control model complexity and ensure smooth optimization.      | Provides smooth gradients, improves generalization by keeping weights small, and is computationally efficient. | Does not yield sparse solutions; models remain dense.                      |
| **L1L2 (Elastic Net)**  | Combines L1 and L2 penalties to balance sparsity and weight decay.                                         | Use when you need both sparsity and stability, particularly with correlated features.             | Balances feature selection and smooth optimization; hyperparameters allow flexible tuning.               | Increases complexity in hyperparameter tuning; requires balancing two penalties. |
| **Orthogonal**          | Adds a penalty that encourages weight matrices to be orthogonal (penalizing the deviation of \(W^T W\) from the identity). | Ideal for deep or recurrent networks where diverse features and stable gradient flow are crucial. | Promotes diversity among neurons, reduces redundancy, and improves gradient flow in deep architectures.    | Computationally more expensive and adds extra hyperparameter tuning requirements.  |

In [ ]:
def build_cnn(
    trial: Any,
    x: layers.Layer,
    max_layers: int = 5,
    max_kernel: int = 10,
    max_filters: int = 256,
    min_filters: int = 32,
    step: int = 32,
    max_decay_factor: float = 0.9,
    use_regularization: bool = False,
    residual_method: Optional[str] = None,
    custom_name: str = "cnn",
) -> layers.Layer:
    """
    Builds the CNN funnel.

    Args:
        trial (Any): The optimization trial containing hyperparameters.
        x (layers.Layer): The input tensor (e.g., reshaped LiDAR data).
        max_layers (int): Maximum number of convolutional layers.
        max_kernel (int): Maximum kernel size for convolutional layers.
        max_filters (int): Maximum number of filters for convolutional layers.
        min_filters (int): Minimum number of filters for convolutional layers.
        step (int): Step size for filters.
        max_decay_factor (float): Maximum decay factor for filters and kernel size.
        use_regularization (bool): Whether to apply regularization on the layers.
        residual_method (Optional[str]): Type of residual connection method to use.
        custom_name (str): Custom prefix for layer names.

    Returns:
        layers.Layer: The processed output tensor.
    """
    cnn_layers = trial.suggest_int(f"{custom_name}_layers", 1, max_layers)
    cnn_filters_layer_0 = trial.suggest_int(f"{custom_name}_filters_layer_0", min_filters, max_filters, step=step)
    filters_decay_factor = trial.suggest_float(f"{custom_name}_filter_decay_factor", 0.0, max_decay_factor, step=0.1)
    kernel_decay_factor = trial.suggest_float(f"{custom_name}_kernel_decay_factor", 0.0, max_decay_factor, step=0.1)

    residual_cnn = None
    skip_connections_cnn = []

    for i in range(cnn_layers):
        filters = (
            cnn_filters_layer_0 if i == 0 else max(16, int(cnn_filters_layer_0 * (filters_decay_factor**i)))
        )
        
        kernel_limit = int(max_kernel * (kernel_decay_factor ** i))
        kernel_size = (
            trial.suggest_int(f"{custom_name}_kernel_height_{i}", 1, max(1, kernel_limit)),
            trial.suggest_int(f"{custom_name}_kernel_width_{i}", 1, max(1, kernel_limit))
        )

        activation = trial.suggest_categorical(
            f"{custom_name}_activation_layer_{i}",
            [
                "relu",
                "tanh",
                "sigmoid",
                "elu",
                "swish",
                "leaky_relu",
            ],
        )

        cnn_kernel_regularizer = (
            get_regularizer(trial, f"cnn_kernel_regularizer_layer_{i}") if use_regularization else None
        )
        cnn_bias_regularizer = (
            get_regularizer(trial, f"cnn_bias_regularizer_layer_{i}") if use_regularization else None
        )
        cnn_activity_regularizer = (
            get_regularizer(trial, f"cnn_activity_regularizer_layer_{i}") if use_regularization else None
        )
        x = layers.Conv2D(
            filters=filters,
            kernel_size=kernel_size,
            activation=activation,
            padding="same",
            name=f"{custom_name}_conv2d_{i}",
            kernel_regularizer=cnn_kernel_regularizer,
            bias_regularizer=cnn_bias_regularizer,
            activity_regularizer=cnn_activity_regularizer,
        )(x)

        if trial.suggest_categorical(f"{custom_name}_use_batch_norm_layer_{i}", [True, False]):
            x = layers.BatchNormalization(name=f"{custom_name}_batch_norm_{i}")(x)

        # ------------------------- Residual Connection Logic ------------------------ #
        if residual_method == "beside":
            if i == 0:
                residual_cnn = x
            else:
                if trial.suggest_categorical(f"{custom_name}_use_residual_layer_{i}", [True, False]):
                    target_channels = x.shape[-1]

                    if residual_cnn.shape[-1] != target_channels:
                        residual_cnn = layers.Conv2D(
                            filters=target_channels,
                            kernel_size=(1, 1),
                            padding="same",
                            activation=None,
                            name=f"{custom_name}_residual_conv2d_{i}",
                        )(residual_cnn)

                    x = layers.Add()([x, residual_cnn])
                    residual_cnn = x
                else:
                    residual_cnn = x

        elif residual_method == "all":
            if i == 0:
                skip_connections_cnn = [x]
            else:
                residuals_to_add = []
                for j, prev in enumerate(skip_connections_cnn):
                    if trial.suggest_categorical(f"{custom_name}_use_residual_layer_{i}_{j}", [True, False]):
                        target_channels = x.shape[-1]
                        adjusted_prev = prev

                        if adjusted_prev.shape[-1] != target_channels:
                            adjusted_prev = layers.Conv2D(
                                filters=target_channels,
                                kernel_size=(1, 1),
                                padding="same",
                                activation=None,
                                name=f"{custom_name}_skip_residual_conv2d_{i}_{j}",
                            )(adjusted_prev)

                        residuals_to_add.append(adjusted_prev)

                if residuals_to_add:
                    x = layers.Add()([x] + residuals_to_add)
                skip_connections_cnn.append(x)

    return x

In [ ]:
def build_dnn(
    trial: Any,
    x: layers.Layer,
    max_layers: int = 10,
    max_units: int = 2048,
    min_units: int = 128,
    step: int = 128,
    max_decay_factor: float = 0.9,
    use_regularization: bool = False,
    residual_method: Optional[str] = None,
    custom_name: str = "dnn",
) -> layers.Layer:
    """
    Builds the DNN funnel.

    Args:
        trial (Any): The optimization trial containing hyperparameters.
        x (layers.Layer): The input tensor (e.g., GPS coordinate data).
        max_layers (int): Maximum number of layers in the DNN.
        max_units (int): Maximum number of units in a layer.
        min_units (int): Minimum number of units in a layer.
        step (int): Step size for the number of units.
        max_decay_factor (float): Maximum decay factor for reducing units in subsequent layers.
        use_regularization (bool): Whether to apply regularization on the layers.
        residual_method (Optional[str]): Type of residual connection method to use.
        custom_name (str): Custom prefix for layer names.

    Returns:
        layers.Layer: The processed output tensor.
    """
    dnn_layers = trial.suggest_int(f"{custom_name}_layers", 1, max_layers)
    units_layer_0 = trial.suggest_int(f"{custom_name}_units_layer_0", min_units, max_units, step=step)
    decay_factor = trial.suggest_float(f"{custom_name}_decay_factor", 0.0, max_decay_factor, step=0.1)

    residual_dense = None
    skip_connections_dense = []

    for i in range(dnn_layers):
        units = units_layer_0 if i == 0 else max(16, int(units_layer_0 * (decay_factor**i)))

        activation = trial.suggest_categorical(
            f"{custom_name}_activation_layer_{i}",
            [
                "relu",
                "tanh",
                "sigmoid",
                "linear",
                "swish",
                "softplus",
                "leaky_relu",
            ],
        )
        
        dnn_kernel_regularizer = (
            get_regularizer(trial, f"dnn_kernel_regularizer_layer_{i}") if use_regularization else None
        )
        dnn_bias_regularizer = (
            get_regularizer(trial, f"dnn_bias_regularizer_layer_{i}") if use_regularization else None
        )
        dnn_activity_regularizer = (
            get_regularizer(trial, f"dnn_activity_regularizer_layer_{i}") if use_regularization else None
        )
        x = layers.Dense(
            units=units,
            activation=activation,
            name=f"{custom_name}_dense_{i}",
            kernel_regularizer=dnn_kernel_regularizer,
            bias_regularizer=dnn_bias_regularizer,
            activity_regularizer=dnn_activity_regularizer,
        )(x)

        if trial.suggest_categorical(f"{custom_name}_use_batch_norm_layer_{i}", [True, False]):
            x = layers.BatchNormalization(name=f"{custom_name}_batch_norm_{i}")(x)

        dropout_rate = trial.suggest_float(f"{custom_name}_dropout_layer_{i}", 0.0, 0.5, step=0.1)
        x = layers.Dropout(dropout_rate, name=f"{custom_name}_dropout_{i}")(x)

        # ------------------------- Residual Connection Logic ------------------------ #
        if residual_method == "beside":
            if i == 0:
                residual_dense = x
            else:
                if trial.suggest_categorical(f"{custom_name}_use_residual_layer_{i}", [True, False]):
                    if residual_dense.shape[-1] != x.shape[-1]:
                        residual_dense = layers.Dense(
                            units=x.shape[-1], activation=None, name=f"{custom_name}_residual_dense_{i}"
                        )(residual_dense)
                    x = layers.Add(name=f"{custom_name}_residual_add_{i}")([x, residual_dense])
                    residual_dense = x
                else:
                    residual_dense = x

        elif residual_method == "all":
            if i == 0:
                skip_connections_dense = [x]
            else:
                residuals_to_add = []
                for j, prev in enumerate(skip_connections_dense):
                    if trial.suggest_categorical(f"{custom_name}_use_residual_layer_{i}_{j}", [True, False]):
                        adjusted_prev = prev

                        if adjusted_prev.shape[-1] != x.shape[-1]:
                            adjusted_prev = layers.Dense(
                                units=x.shape[-1],
                                activation=None,
                                name=f"{custom_name}_skip_residual_dense_{i}_{j}",
                            )(adjusted_prev)

                        residuals_to_add.append(adjusted_prev)
                if residuals_to_add:
                    x = layers.Add(name=f"{custom_name}_add_{i}")([x] + residuals_to_add)
                skip_connections_dense.append(x)

    return x

In [ ]:
def objective(
    trial: optuna.Trial,
    x_lidar_train: np.ndarray,
    x_coord_train: np.ndarray,
    y_train: np.ndarray,
    x_lidar_val: np.ndarray,
    x_coord_val: np.ndarray,
    y_val: np.ndarray,
    checkpoint_path: str,
    model_path: str,
    fig_dir: str,
    epochs: int = 50,
    size_penalizer: str = None,
    use_regularization: bool = False,
    residual_method: str = None,
) -> float:
    """
    Objective function for optimizing a Neural Network (NN).

    Args:
        trial (optuna.Trial): A single trial of the optimization process.
        x_lidar_train (np.ndarray): Training data for LiDAR input.
        x_coord_train (np.ndarray): Training data for GPS coordinates.
        y_train (np.ndarray): Training labels.
        x_lidar_val (np.ndarray): Validation data for LiDAR input.
        x_coord_val (np.ndarray): Validation data for GPS coordinates.
        y_val (np.ndarray): Validation labels.
        checkpoint_path (str): Path to a checkpoint file.
        model_path (str): Path to save the model.
        fig_dir (str): Directory to save the figures.
        epochs (int, optional): Number of training epochs. Defaults to 50.
        size_penalizer (str, optional): Type of size penalizer ("flops" or "params"). Defaults to None.
        use_regularization (bool, optional): Whether to use regularization. Defaults to False.
        residual_method (str, optional): Method for residual connections ("beside", "all", or None). Defaults to None.

    Returns:
        float: The best validation loss achieved during this trial.
    """

    # ------------------------------- Scaler & PCA ------------------------------- #
    # Apply scaler (and optionally PCA) on the coords
    scaler_name = trial.suggest_categorical(
        "scaler",
        [
            "StandardScaler",
            "MinMaxScaler_0_1",
            "MinMaxScaler_-1_1",
        ],
    )
    if scaler_name == "StandardScaler":
        scaler = StandardScaler()
    elif scaler_name == "MinMaxScaler_0_1":
        scaler = MinMaxScaler(feature_range=(0, 1))
    elif scaler_name == "MinMaxScaler_-1_1":
        scaler = MinMaxScaler(feature_range=(-1, 1))

    x_coord_train = scaler.fit_transform(x_coord_train)
    x_coord_val = scaler.transform(x_coord_val)

    # ------------------------------- LiDAR Branch ------------------------------- #
    # Input for LiDAR data (e.g., shape: (20, 200, 10))
    x_lidar_input = layers.Input(shape=(20, 200, 10))

    x = build_cnn(
        trial=trial,
        x=x_lidar_input,
        max_layers=5,
        max_kernel=5,
        max_filters=128,
        min_filters=32,
        step=32,
        use_regularization=use_regularization,
        residual_method=residual_method,
        custom_name="lidar",
    )

    x_lidar_output = layers.Flatten()(x)

    # -------------------------------- GPS Branch -------------------------------- #
    # Input for coordinate data (e.g., shape: (2,))
    x_coord_input = layers.Input(shape=(x_coord_train.shape[1],))

    x_coord_output = build_dnn(
        trial=trial,
        x=x_coord_input,
        max_layers=10,
        max_units=1024,
        min_units=128,
        step=128,
        max_decay_factor=0.9,
        use_regularization=use_regularization,
        residual_method=residual_method,
        custom_name="coord",
    )

    # ----------------------------- Combine Branches ----------------------------- #
    combined = layers.Concatenate()([x_lidar_output, x_coord_output])

    extra = build_dnn(
        trial=trial,
        x=combined,
        max_layers=2,
        max_units=512,
        min_units=64,
        step=64,
        max_decay_factor=0.9,
        use_regularization=False,
        residual_method=None,
        custom_name="extra",
    )

    # ---------------------------------- Output ---------------------------------- #
    outputs = layers.Dense(256, activation="softmax")(extra)

    # ----------------------------------- Model ---------------------------------- #
    model = Model(inputs=(x_lidar_input, x_coord_input), outputs=(outputs,))
    
    # print(f"Model summary: {model.summary()}")

    optimizer_name = trial.suggest_categorical(
        "optimizer",
        [
            "Adam",
            "RMSprop",
            "SGD",
            "AdamW",
            "Nadam",
            "Lion",
        ],
    )
    learning_rate = trial.suggest_float("learning_rate", 1e-4, 1e-2, log=True)
    if optimizer_name == "Adam":
        optimizer = optimizers.Adam(learning_rate=learning_rate)
    elif optimizer_name == "RMSprop":
        optimizer = optimizers.RMSprop(learning_rate=learning_rate)
    elif optimizer_name == "SGD":
        optimizer = optimizers.SGD(learning_rate=learning_rate)
    elif optimizer_name == "AdamW":
        optimizer = optimizers.AdamW(learning_rate=learning_rate)
    elif optimizer_name == "Nadam":
        optimizer = optimizers.Nadam(learning_rate=learning_rate)
    elif optimizer_name == "Lion":
        optimizer = optimizers.Lion(learning_rate=learning_rate)

    model.compile(optimizer=optimizer, loss=losses.SparseCategoricalCrossentropy(), metrics=["accuracy"])

    # ------------------------------- Callbacks -------------------------------
    early_stopping = callbacks.EarlyStopping(
        monitor="val_loss", patience=6, restore_best_weights=True, mode="min"
    )
    reduce_lr = callbacks.ReduceLROnPlateau(monitor="val_loss", patience=3)
    checkpoint_callback = callbacks.ModelCheckpoint(
        filepath=os.path.join(checkpoint_path, f"trial_{trial.number}_best_model.weights.h5"),
        save_weights_only=True,
        monitor="val_loss",
        mode="min",
        save_best_only=True,
        verbose=0,
    )

    # ------------------------------- Train Model -------------------------------
    history = model.fit(
        [x_lidar_train, x_coord_train],
        y_train,
        epochs=epochs,
        batch_size=trial.suggest_categorical("batch_size", [32, 64, 128, 256]),
        validation_data=([x_lidar_val, x_coord_val], y_val),
        callbacks=[early_stopping, reduce_lr, checkpoint_callback],
        verbose=2,
    )
    
    model.save(os.path.join(model_path, f"trial_{trial.number}_model.keras"))

    # ------------------------- Penalize the Model Size -------------------------
    param_count = model.count_params()
    best_val_loss = min(history.history["val_loss"])
    if size_penalizer == "flops":
        forward_pass = tf.function(
            model.call, input_signature=[tf.TensorSpec(shape=(1,) + model.input_shape[1:])]
        )
        # Ensure ProfileOptionBuilder and profile are imported as needed.
        graph_info = profile(
            forward_pass.get_concrete_function().graph, options=ProfileOptionBuilder.float_operation()
        )
        flops = graph_info.total_float_ops
        lambda_flops = 1e-10  # Example value
        loss = best_val_loss + lambda_flops * flops
    elif size_penalizer == "params":
        lambda_param = 1e-9  # Example value
        loss = best_val_loss + lambda_param * param_count
    else:
        loss = best_val_loss
        
    # ----------------------------------- Plots ---------------------------------- #
    clear_output(wait=True)
    print("------------------------------ Last Trial Info -----------------------------")
    try:
        from utils import plot_api

        # Plot Loss
        plot_api.plot_scientific(
            x=range(1, len(history.history["loss"]) + 1),
            y_datasets=[
                history.history["loss"], 
                history.history["val_loss"]
            ],
            labels=[
                "Training Loss", 
                "Validation Loss"
            ],
            x_label="Epoch",
            y_label="Loss",
            title="Training & Validation Loss",
            markers=["o", "x"],
            line_styles=["-", "--"],
            ylim=(0, max(max(history.history["loss"]), max(history.history["val_loss"])) * 1.05),
            grid=True,
            x_integer=True,
            save_path=os.path.join(fig_dir, f"trial_{trial.number}_loss_plot.png"),
        )

        # Plot Accuracy
        plot_api.plot_scientific(
            x=range(1, len(history.history["accuracy"]) + 1),
            y_datasets=[
                history.history["accuracy"], 
                history.history["val_accuracy"]
            ],
            labels=[
                "Training Accuracy", 
                "Validation Accuracy"
            ],
            x_label="Epoch",
            y_label="Accuracy",
            title="Training & Validation Accuracy",
            markers=["v", "^"],
            line_styles=["-", "--"],
            ylim=(0, 1),
            grid=True,
            x_integer=True,
            save_path=os.path.join(fig_dir, f"trial_{trial.number}_accuracy_plot.png"),
        )
    except Exception as e:
        print("[ERROR] Failed to save loss plot.")
        print(e)

    print(f"Trial loss: {loss}")
    print(f"PARAMS: {param_count}")
    print("----------------------------------------------------------------------------")

    clear_session()
    del model

    return loss

## 6. Do the NAS

### 6.1. Code Health Check

In [ ]:
# ------------------------------- Log Resources ------------------------------ #
#! Remove this block if you don't want to use it
# try:
#     from utils import log_resources
#     LOG_DIR = os.path.join(RUN_DIR, "logs")
#     os.makedirs(LOG_DIR, exist_ok=True)
    
#     log_resources.log_resources(
#         log_dir=LOG_DIR,
#         interval=5,
#         cpu=False,
#         ram=True,
#         gpu=True,
#         cuda=False,
#         tensorflow=False,
#     )
# except Exception as e:
#     print("[ERROR] Failed to log resources!")
#     print(e)
#     pass

In [ ]:
# ---------------------------- Kernel life monitor --------------------------- #
#! Remove this block if you don't want to use it
try:
    pid = os.getpid()
    display(HTML(f'Call the monitor script: <span style="color: orange;">python _monitor_kernel_life.py --pid {pid}</span>'))
except Exception as e:
    print("[ERROR] Kernel monitoring failed!")
    print(e)
    pass

### 6.2. Main

In [ ]:
# ----------------------------- Hyperparameters ------------------------------ #
NUM_TRIALS = 200
EPOCHS = 50
TOP_K = 10  # Number of top trials to save

In [ ]:
def optimize() -> None:
    """
    Runs an Optuna study for optimizing the Neural Network.
    """
    try:
        print("Starting the optimization process...")

        # ------------------------------- Storage paths ------------------------------ #
        study_name = os.path.join(RUN_DIR, "optuna_study")
        os.makedirs(study_name, exist_ok=True)

        args_dir = os.path.join(study_name, "args")
        os.makedirs(args_dir, exist_ok=True)

        fig_dir = os.path.join(study_name, "figures")
        os.makedirs(fig_dir, exist_ok=True)

        storage_path = f"sqlite:///{os.path.join(study_name, 'optuna_study.db')}"
        checkpoint_path = os.path.join(study_name, "weights")
        os.makedirs(checkpoint_path, exist_ok=True)

        model_path = os.path.join(study_name, "models")
        os.makedirs(model_path, exist_ok=True)

        print(f"Initializing study with name '{study_name}'...")

        # ---------------------------------- Pruners --------------------------------- #
        pruner = optuna.pruners.HyperbandPruner()

        # ----------------------------------- Study ---------------------------------- #
        study = optuna.create_study(
            study_name=study_name,
            storage=storage_path,
            direction="minimize",
            pruner=pruner,
            load_if_exists=True
        )

        study.optimize(
            lambda trial: objective(
                trial,
                x_lidar_train=s008_lidar_input,
                x_coord_train=s008_coord_input,
                y_train=s008_y_train,
                x_lidar_val=s009_lidar_input,
                x_coord_val=s009_coord_input,
                y_val=s009_y_train,
                checkpoint_path=checkpoint_path,
                model_path=model_path,
                fig_dir=fig_dir,
                epochs=EPOCHS,
                size_penalizer=None,
                use_regularization=True,
                residual_method="all",
            ),
            n_trials=NUM_TRIALS,
            catch=(Exception,),
            gc_after_trial=True,
            show_progress_bar=True,
        )

        # ----------------------------- Save Top-K Trials ---------------------------- #
        valid_trials = [t for t in study.trials if t.value is not None]
        sorted_trials = sorted(valid_trials, key=lambda t: t.value)[:TOP_K]

        for rank, trial in enumerate(sorted_trials):
            trial_id = trial.number
            trial_params = trial.params
            trial_loss = trial.value

            save_trial_params_to_file(
                filepath=os.path.join(args_dir, f"top_{rank + 1}_trial.txt"),
                params=trial_params,
                rank=rank + 1,
                trial_id=trial_id,
                loss=trial_loss,
                sampler=study.sampler.__class__.__name__,
            )

        # ------------------------ Clean-Up Non-Top Trials --------------------------- #
        all_trial_ids = {t.number for t in study.trials}
        top_trial_ids = {t.number for t in sorted_trials}

        for trial_id in all_trial_ids - top_trial_ids:
            weights_path = os.path.join(checkpoint_path, f"trial_{trial_id}_best_model.weights.h5")
            model_file_path = os.path.join(model_path, f"trial_{trial_id}_model.keras")
            mse_plot_path = os.path.join(fig_dir, f"trial_{trial_id}_mse_plot.png")
            accuracy_plot_path = os.path.join(fig_dir, f"trial_{trial_id}_accuracy_plot.png")

            if os.path.exists(weights_path):
                os.remove(weights_path)
            if os.path.exists(model_file_path):
                os.remove(model_file_path)
            if os.path.exists(mse_plot_path):
                os.remove(mse_plot_path)
            if os.path.exists(accuracy_plot_path):
                os.remove(accuracy_plot_path)

        print(f"Saved top-{TOP_K} trials.\n")

        analyze_study(study, fig_dir)

    except Exception as e:
        print(f"An error occurred: {e}")
        traceback.print_exc()

In [ ]:
run_with_notification(
        func=optimize,
        func_args=(),
        func_kwargs={},
        recipients_file="./json/recipients.json",
        credentials_file="./json/credentials.json",
        subject_success=f"✅ Top-K-Beam-Selection-ML Training Complete",
        body_success="""
            <html>
                <body style="font-family: Arial, sans-serif; line-height: 1.6; color: #333; background-color: #f9f9f9; padding: 20px;">
                <div style="max-width: 600px; margin: auto; background: #fff; padding: 20px; border: 1px solid #ddd; border-radius: 8px;">
                <h2 style="color: #0056b3; text-align: center;">🎉 Study Complete</h2>
                <p style="font-size: 16px; color: #444;">
                <strong>Dear User,</strong>
                </p>
                <p style="font-size: 18px; color: #333;">
                Your study has successfully completed!
                </p>
                <p style="font-size: 16px; color: #555;">
                The top-k trials were saved.
                </p>
                <p style="text-align: center; font-size: 16px;">
                <strong style="color: #28a745;">✔️ Study Status:</strong> <span style="color: #0056b3;">Completed</span>
                </p>
                <footer style="margin-top: 20px; text-align: center; font-size: 14px; color: #888;">
                <p>Best regards,</p>
                <p><strong>The Optimization Team</strong></p>
                </footer>
                </div>
                </body>
            </html>
            """,
        text_type="html",
    )